- 把 prompt 喂给 Actor → 生成 response
- 把完整 "prompt + response" 喂给 Ref 模型 → 拿到 logits / log_prob / KL
- 打印数值，一眼看懂二者差异

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 加载两个同规模但权重不同的模型
tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
actor = AutoModelForCausalLM.from_pretrained("gpt2")      # 训练中的策略
ref = AutoModelForCausalLM.from_pretrained("gpt2")        # 参考策略（固定）

# 2. Actor 生成
prompt = "The answer is"
inputs = tok(prompt, return_tensors='pt')
with torch.no_grad():
    output = actor.generate(**inputs, max_new_tokens=10, do_sample=True, temperature=0.7,
                            return_dict_in_generate=True, output_scores=True)
response_ids = output.sequences[:, inputs.input_ids.shape[1]:]  # 只取新生成部分
response_txt = tok.decode(response_ids[0], skip_special_tokens=True)

# 3. 拼成 "prompt + response" 再喂给 Ref
full_ids = output.sequences  # 已包含 prompt + response
with torch.no_grad():
    logits_ref = ref(full_ids).logits[:, -1, :]          # 最后一个位置
    logits_act = actor(full_ids).logits[:, -1, :]

# 4. 计算 KL 项（单位置示例）
logp_ref = logits_ref.log_softmax(-1)
logp_act = logits_act.log_softmax(-1)
kl_token = (logp_ref - logp_act).gather(1, full_ids[:, -1].unsqueeze(1))

print("prompt        :", repr(prompt))
print("actor response:", repr(response_txt))
print("log π_ref(last):", logp_ref[0, full_ids[0, -1]].item())
print("log π_act(last):", logp_act[0, full_ids[0, -1]].item())
print("KL(last token) :", kl_token.item(), "≈ log(π_ref/π_θ)")

一句话看懂
把 "prompt + actor_response" 整体再喂给 Ref 模型，就能拿到 π_ref 的 log_prob，与 π_θ 对比后得到 KL 惩罚项——这就是 RLHF 里约束策略更新的核心数值。

## logits_ref和logits_act对应的token id 是一样的吗？
- 是的，token id 完全一样！

两条 logits 对应的是同一段 token 序列（prompt + response），只是由两个不同模型分别前向计算得到：
- full_ids 形状 [1, L]
- logits_ref & logits_act 形状 [1, L, vocab]
- 在相同位置 t 用 同一个 token id 去 gather

In [ ]:
logp_ref = logits_ref[:, t, :].log_softmax(-1)[token_id]
logp_act = logits_act[:, t, :].log_softmax(-1)[token_id]

### token_id 不变，变的是概率值——这正是 KL 惩罚要量的“策略漂移”。